## Final Project Submission

Please fill out:
* Student name: George Nyandusi
* Student pace: full time
* Scheduled project review date/time: 
* Instructor name: Nikita Njoroge
* Blog post URL:


In [19]:
# Your code here - remember to use markdown cells for comments as well!

### Business Problem
Your company now sees all the big companies creating original video content and they want to get in on the fun. They have decided to create a new movie studio, but they don’t know anything about creating movies. You are charged with exploring what types of films are currently doing the best at the box office. You must then translate those findings into actionable insights that the head of your company's new movie studio can use to help decide what type of films to create.

### Objectives
 1. To Investigate the genres, themes, and formats of films that are performing well at the box office.
 2. To Benchmark Against Competitors: Compare the strategies of leading movie studios to identify gaps and opportunities for   your company to differentiate itself in the market
 3. Develop Actionable Recommendations: Translate findings into strategic insights, such as which genres to prioritize, potential partnerships, or innovative approaches to sales and distribution.

### IMDB

In [20]:
# import necessary libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import zipfile

In [21]:
#Defining the path to the zip file and the CSV file inside it
with zipfile.ZipFile('zippedData/im.db.zip', 'r') as zip_ref:
    zip_ref.extractall('im.db')

In [22]:
#connecting to the SQLite database
conn = sqlite3.connect('im.db/im.db')
print("Connection successful!")

Connection successful!


In [23]:
# Querying the database to get the table names
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
tables

[('movie_basics',),
 ('directors',),
 ('known_for',),
 ('movie_akas',),
 ('movie_ratings',),
 ('persons',),
 ('principals',),
 ('writers',)]

In [24]:
# Query to select movies released after 2010
query = """
SELECT movie_id, primary_title, start_year
FROM movie_basics
WHERE start_year >= 2010;
"""
cursor.execute(query)

# Fetch and print results
results = cursor.fetchall()
for row in results:
    print(row)

('tt0063540', 'Sunghursh', 2013)
('tt0066787', 'One Day Before the Rainy Season', 2019)
('tt0069049', 'The Other Side of the Wind', 2018)
('tt0069204', 'Sabse Bada Sukh', 2018)
('tt0100275', 'The Wandering Soap Opera', 2017)
('tt0111414', 'A Thin Life', 2018)
('tt0112502', 'Bigfoot', 2017)
('tt0137204', 'Joe Finds Grace', 2017)
('tt0139613', 'O Silêncio', 2012)
('tt0144449', 'Nema aviona za Zagreb', 2012)
('tt0146592', 'Pál Adrienn', 2010)
('tt0154039', 'So Much for Justice!', 2010)
('tt0159369', 'Cooper and Hemingway: The True Gen', 2013)
('tt0162942', 'Children of the Green Dragon', 2010)
('tt0170651', 'T.G.M. - osvoboditel', 2018)
('tt0176694', 'The Tragedy of Man', 2011)
('tt0187902', "How Huang Fei-hong Rescued the Orphan from the Tiger's Den", 2011)
('tt0192528', 'Heaven & Hell', 2018)
('tt0230212', 'The Final Journey', 2010)
('tt0247643', 'Los pájaros se van con la muerte', 2011)
('tt0249516', 'Foodfight!', 2012)
('tt0250404', 'Godfather', 2012)
('tt0253093', 'Gangavataran', 201

In [25]:
# Inspect the structure of a specific table
cursor.execute("PRAGMA table_info(movie_basics);")
columns = cursor.fetchall()

# Convert the results into a Pandas DataFrame for better readability
columns_df = pd.DataFrame(columns, columns=["cid", "name", "type", "notnull", "dflt_value", "pk"])

print(columns_df)


   cid             name     type  notnull dflt_value  pk
0    0         movie_id     TEXT        0       None   0
1    1    primary_title     TEXT        0       None   0
2    2   original_title     TEXT        0       None   0
3    3       start_year  INTEGER        0       None   0
4    4  runtime_minutes     REAL        0       None   0
5    5           genres     TEXT        0       None   0


In [26]:
# Combine data from the movie_basics and movie_ratings tables
# to analyze the relationship between genres and ratings
query = """
SELECT mb.genres, mr.averagerating, mr.numvotes
FROM movie_basics mb
JOIN movie_ratings mr
ON mb.movie_id = mr.movie_id
"""
genres_ratings_df = pd.read_sql_query(query, conn)
genres_ratings_df.head(10)


,genres,averagerating,numvotes
0,"Action,Crime,Drama",7.00,77
1,"Biography,Drama",7.20,43
2,Drama,6.90,4517
3,"Comedy,Drama",6.10,13
4,"Comedy,Drama,Fantasy",6.50,119
5,"Horror,Thriller",4.10,32
6,"Adventure,Animation,Comedy",8.10,263
7,Drama,6.80,451
8,History,4.60,64
9,Documentary,7.60,53


In [27]:
# Query to link directors with their associated movies and ratings
query = """
SELECT d.person_id AS director_id, mb.primary_title, mb.genres, mr.averagerating, mr.numvotes
FROM directors d
JOIN movie_basics mb ON d.movie_id = mb.movie_id
JOIN movie_ratings mr ON mb.movie_id = mr.movie_id
"""
directors_ratings_df = pd.read_sql_query(query, conn)

# Display the DataFrame with better formatting
pd.set_option('display.max_columns', None)  
pd.set_option('display.max_rows', 10)       
pd.set_option('display.float_format', '{:.2f}'.format)  # Format floats to 2 decimal places

# Display the DataFrame
directors_ratings_df.head(10)  

,director_id,primary_title,genres,averagerating,numvotes
0,nm0899854,Life's a Beach,Comedy,3.90,219
1,nm1940585,Steve Phoenix: The Untold Story,Drama,5.50,18
2,nm0151540,The Babymakers,Comedy,5.00,8147
3,nm0151540,The Babymakers,Comedy,5.00,8147
4,nm0089502,Bulletface,Thriller,5.80,875
5,nm2291498,Bulletface,Thriller,5.80,875
6,nm2292011,Bulletface,Thriller,5.80,875
7,nm2416460,Torn,Thriller,6.80,21
8,nm2286991,Legend of the Red Reaper,"Action,Adventure,Fantasy",2.20,495
9,nm2286991,Legend of the Red Reaper,"Action,Adventure,Fantasy",2.20,495


In [28]:
# Query to extract release years and associated ratings
query = """
SELECT mb.start_year AS release_year, mr.averagerating, mr.numvotes
FROM movie_basics mb
JOIN movie_ratings mr ON mb.movie_id = mr.movie_id
WHERE mb.start_year IS NOT NULL
"""
trends_over_time_df = pd.read_sql_query(query, conn)

# Display the DataFrame with better formatting
pd.set_option('display.max_columns', None)  
pd.set_option('display.max_rows', 10)       
pd.set_option('display.float_format', '{:.2f}'.format) 

trends_over_time_df.head(10)

,release_year,averagerating,numvotes
0,2013,7.00,77
1,2019,7.20,43
2,2018,6.90,4517
3,2018,6.10,13
4,2017,6.50,119
5,2017,4.10,32
6,2017,8.10,263
7,2010,6.80,451
8,2010,4.60,64
9,2013,7.60,53


### The numbers dataset

In [29]:
# import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import zipfile

In [30]:
#loading the Numbers dataset
df = pd.read_csv("./zippedData/tn.movie_budgets.csv.gz")
df.head(20) 

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"
...,...,...,...,...,...,...
15,16,"May 4, 2007",Spider-Man 3,"$258,000,000","$336,530,303","$894,860,230"
16,17,"May 6, 2016",Captain America: Civil War,"$250,000,000","$408,084,349","$1,140,069,413"
17,18,"Mar 25, 2016",Batman v Superman: Dawn of Justice,"$250,000,000","$330,360,194","$867,500,281"
18,19,"Dec 14, 2012",The Hobbit: An Unexpected Journey,"$250,000,000","$303,003,568","$1,017,003,568"


In [31]:
# Define relevant columns for analysis
relevant_columns = ['movie', 'domestic_gross', 'worldwide_gross', 'production_budget', 'release_date']
df = df[relevant_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5782 entries, 0 to 5781
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   movie              5782 non-null   object
 1   domestic_gross     5782 non-null   object
 2   worldwide_gross    5782 non-null   object
 3   production_budget  5782 non-null   object
 4   release_date       5782 non-null   object
dtypes: object(5)
memory usage: 226.0+ KB


This is to Understand the structure of your DataFrame.¶
Checking for missing values (via Non-Null Count).
Verifying column data types.
Analyzing memory usage when working with large datasets

In [32]:
#checking the shape of the dataset  
df.shape

AttributeError: 'NoneType' object has no attribute 'shape'

### checking for missing values

In [ ]:
# Checking for missing values
df.isnull().sum()

In [ ]:
# Describing the dataset to get a summary of statistics
df.describe() 

### checking for duplicates

In [ ]:
# Checking for duplicates and dropping them if any
df = pd.read_csv("./zippedData/tn.movie_budgets.csv.gz")
df.duplicated().sum()
df.drop_duplicates()

In [ ]:
# inspecting the dataset for any outliers
df.describe(include='all')

### Data exploration conclusion
The dataset highlights trends in movie performance, with notable clusters in release dates and titles, challenges like missing and non-numeric data requiring cleaning, and opportunities for deeper analysis into profits, genres, and seasonal impacts over the past 15 years

### Data cleaning 
Data cleaning will be necessary in our dataset to convert columns like production_budget, domestic_gross and worldwide_gross from non-numeric formats (e.g., $ and commas) into usable numeric data, address missing or zero-dollar values that might skew analysis and ensure accurate profit calculations and meaningful insights into trends like genre performance and seasonal impacts. Clean data will enable reliable analysis and will help uncover actionable insights without being misled by errors or inconsistencies.

In [ ]:
# Filtering the dataset for movies released after 2010
df['release_date'] = pd.to_datetime(df['release_date'])
df = df[df['release_date'] >= '2010-01-01']
df.head(10)

In [ ]:
#Checking the data types of the columns
df.dtypes
df['production_budget'] = df['production_budget'].replace('[\$,]', '', regex=True).astype(float)
df['domestic_gross'] = df['domestic_gross'].replace('[\$,]', '', regex=True).astype(float)
df['worldwide_gross'] = df['worldwide_gross'].replace('[\$,]', '', regex=True).astype(float)
df['production_budget'] = df['production_budget'].astype(float)
df['domestic_gross'] = df['domestic_gross'].astype(float)
df['worldwide_gross'] = df['worldwide_gross'].astype(float)

In [ ]:
# Handling missing values by filling them with 0
df['production_budget'] = df['production_budget'].fillna(0)
df['domestic_gross'] = df['domestic_gross'].fillna(0)
df['worldwide_gross'] = df['worldwide_gross'].fillna(0)


In [ ]:
# Filtering by relevant dates
df = df[df['release_date'] >= '2010-01-01']
df = df[df['release_date'] <= '2025-12-31']
df.head(10)

In [ ]:
#Add a new column for profit
# Calculate profit as worldwide gross minus production budget
df['profit'] = df['worldwide_gross'] - df['production_budget']
print(df[['movie', 'profit']].sort_values(by='profit', ascending=False).head())

### ROTTEN TOMATOES

In [ ]:
# Importing necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Loading another dataset for analysis
df = pd.read_csv("./zippedData/rt.movie_info.tsv.gz",delimiter="\t", compression="gzip")
df

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Explore the columns in the dataset
df.columns

In [ ]:
# Inspecting missing values
df = df.dropna(subset=['box_office'])
# Replace with statistical values
df['runtime'] = df['runtime'].fillna(df['runtime'].median())
# Flag missing values
df['missing_box_office'] = df['box_office'].isnull()
# Fill with default values
df['currency'].fillna('$', inplace=True)
# Verify the changes
df.isnull().sum()

In [ ]:
# Checking for shape of the dataset
print(df.shape)

In [ ]:
# Standardizing the box office values
df['box_office'] = df['box_office'].replace('[\$,]', '', regex=True).astype(float)
df

In [ ]:
# Cleaning the runtime column
df['runtime'] = df['runtime'].replace('[\D]', '', regex=True).astype(float)
df

In [ ]:
# Format Dates theater date and DVD date
df['theater_date'] = pd.to_datetime(df['theater_date'], errors='coerce')
df['dvd_date'] = pd.to_datetime(df['dvd_date'], errors='coerce')

In [ ]:
for theater_date, dvd_date in zip(df['theater_date'], df['dvd_date']):
    print(f"Theater Date: {theater_date}, DVD Date: {dvd_date}")

In [ ]:
# Deduplicate the dataset based on synopsis, director, and theater date
df = df.drop_duplicates(subset=['synopsis', 'director', 'theater_date'], keep='first')
df

In [ ]:
# Verify Genre Consistency
df['genre'] = df['genre'].str.lower().str.strip()
df

In [ ]:
# Handle missing values in the genre column
df['genre'] = df['genre'].fillna('Unknown')
df['genre'] = df['genre'].replace('', 'Unknown')
df

In [ ]:
# Handle missing values in critical columns
df = df.dropna(subset=['box_office', 'currency', 'runtime'])